# 01 Data Quality

`data/samples/` と `data/processed/` を走査して、読み込めるデータの欠損率・重複・型を確認します。

このノートブックは、まだデータが揃っていなくても止まらないようにしてあります。

In [6]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except Exception:  # pragma: no cover - notebook fallback
    display = print

REPO_ROOT = Path.cwd().resolve()
for candidate in [REPO_ROOT, *REPO_ROOT.parents]:
    if (candidate / 'data').exists() and (candidate / 'src').exists():
        REPO_ROOT = candidate
        break

DATA_DIRS = [REPO_ROOT / 'data' / 'samples', REPO_ROOT / 'data' / 'processed']
SUPPORTED_SUFFIXES = {'.parquet', '.csv', '.json', '.geojson'}

def read_geojson(path: Path) -> pd.DataFrame:
    payload = json.loads(path.read_text(encoding='utf-8'))
    features = payload.get('features', []) if isinstance(payload, dict) else []
    rows: list[dict[str, object]] = []
    for feature in features:
        properties = dict(feature.get('properties') or {})
        geometry = feature.get('geometry') or {}
        properties['geometry_type'] = geometry.get('type')
        properties['geometry'] = geometry.get('coordinates')
        rows.append(properties)
    return pd.DataFrame(rows)

def read_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == '.parquet':
        return pd.read_parquet(path)
    if suffix == '.csv':
        return pd.read_csv(path)
    if suffix in {'.json', '.geojson'}:
        return read_geojson(path)
    raise ValueError(f'Unsupported file: {path}')

def discover_files() -> list[Path]:
    files: list[Path] = []
    for directory in DATA_DIRS:
        if not directory.exists():
            continue
        for path in sorted(directory.rglob('*')):
            if path.is_file() and path.suffix.lower() in SUPPORTED_SUFFIXES:
                files.append(path)
    return files

def quality_summary(frame: pd.DataFrame, name: str) -> dict[str, object]:
    return {
        'dataset': name,
        'rows': int(len(frame)),
        'columns': int(frame.shape[1]),
        'missing_cells': int(frame.isna().sum().sum()),
        'duplicate_rows': int(frame.duplicated().sum()) if not frame.empty else 0,
        'numeric_columns': int(len(frame.select_dtypes(include='number').columns)),
    }

def missing_report(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame(columns=['column', 'missing_count', 'missing_rate'])
    report = pd.DataFrame({
        'column': frame.columns,
        'missing_count': frame.isna().sum().to_numpy(),
        'missing_rate': frame.isna().mean().to_numpy(),
    })
    return report.sort_values(['missing_rate', 'column'], ascending=[False, True]).reset_index(drop=True)

files = discover_files()
frames: dict[str, pd.DataFrame] = {}
for path in files:
    try:
        frames[path.stem] = read_table(path)
    except Exception as exc:
        print(f'failed to read {path.relative_to(REPO_ROOT)}: {exc}')

print(f'repo root: {REPO_ROOT}')
print(f'found {len(files)} candidate file(s), loaded {len(frames)} frame(s)')
for path in files:
    print(f'- {path.relative_to(REPO_ROOT)}')

repo root: /Users/yoshikawahiiragi/Heat-Town/Heat-Town/Heat-Town
found 0 candidate file(s), loaded 0 frame(s)


In [7]:
if not frames:
    print('No readable data found yet. Add files under data/samples/ or data/processed/.')
else:
    overview = pd.DataFrame([quality_summary(frame, name) for name, frame in frames.items()])
    display(overview.sort_values(['rows', 'dataset'], ascending=[False, True]).reset_index(drop=True))

No readable data found yet. Add files under data/samples/ or data/processed/.


In [8]:
for name, frame in frames.items():
    print(f'\n=== {name} ===')
    print(f'shape: {frame.shape}')
    print('columns:', ', '.join(map(str, frame.columns[:30])))
    display(missing_report(frame).head(20))
    if not frame.empty:
        display(frame.head(5))
        numeric = frame.select_dtypes(include='number')
        if not numeric.empty:
            display(numeric.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T)

print('Data quality check complete.')

Data quality check complete.
